# DipDECK on ESM-2 3B, notebook A: seeds 0-4

Notebook A runs seeds 0-4 and notebook B seeds 5-9, at the same time in two Colab sessions. Each
runs the same 20 settings and five GraphPart folds as the ProtT5 raw arm, on half the seeds:

1. Raw 2,560-d reference, 5 fits (fast; checks the whole path first).
2. Raw 2,560-d fold grid, 500 fits.
3. PCA 45 (90% of ESM-2 3B's variance) reference, 5 fits.
4. PCA 45 fold grid, 500 fits; the basis is fitted on each rotation's training folds only.

Each job writes to `DRIVE_ROOT/results/dipdeck_esm2_3b_seeds0to4/` and leaves a ZIP
in `DRIVE_ROOT/`. A job writes nothing until all its fits succeed, and a job whose
output on Drive already validates is skipped, so after a disconnect start a fresh runtime and
**Run all** again. Any failure stops the notebook. The last cell prints every archive and its SHA256.

Upload this project folder to Google Drive as `MyDrive/explaining-plm-latent-spaces` (or set `PROJECT_DIR`). Select **T4 GPU**, use a fresh runtime, and run all cells.


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/explaining-plm-latent-spaces"
CLUSTPY_COMMIT = "82548dc35a01d221474fbb44908b3621d4b97616"
EXPECTED_UPSTREAM_DIPDECK_SHA256 = "c567d24c1722087a75c3eb56f17fe91152b6f818148083d908086b9fc78d438a"
EXPECTED_DIPDECK_SHA256 = "cfbf629ab341bdbb5967af232c24cfc3f2e5fb09cd082fb28b317c45738b7420"
DATA_SHA256 = {
    "data/phosphatase/phosphatase/esm2_t36_3B_UR50D_full_length.h5":
        "00c27d5730ba59db9ca4178118a2ba5fc4e4e4ecc41266afdc9e000951e8571c",
    "data/phosphatase/phosphatase/tmp/prot_t5.h5":
        "425e005d783d70c5ac9043ad2c7d794ce603a8ddbecffa405d569aeddb78c7c1",
}
EMBEDDING = "esm2_3b"
PCA_DIM = 45
SEEDS = "0-4"
DRIVE_ROOT = PROJECT_DIR


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import os
import shutil
import subprocess

repo = Path("/content/project").resolve()
clustpy_source = Path("/content/ClustPy").resolve()
drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)

for target in (repo, clustpy_source):
    if target.parent != Path("/content"):
        raise RuntimeError(f"Refusing to replace unexpected path: {target}")
    if target.exists():
        shutil.rmtree(target)

project_dir = Path(PROJECT_DIR)
if not (project_dir / "src/algo/dipdeck/dipdeck_standalone.py").is_file():
    raise FileNotFoundError(f"Upload this project folder to Google Drive at {project_dir}, then rerun.")
for part in ("src", "data", "results/graphpart", "results/ec_audit"):
    shutil.copytree(project_dir / part, repo / part)
subprocess.run(["git", "clone", "https://github.com/collinleiber/ClustPy.git", str(clustpy_source)], check=True)
subprocess.run(["git", "-C", str(clustpy_source), "checkout", CLUSTPY_COMMIT], check=True)

dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
source_text = dipdeck_source.read_text(encoding="utf-8")
upstream_sha256 = hashlib.sha256(source_text.encode()).hexdigest()
if upstream_sha256 != EXPECTED_UPSTREAM_DIPDECK_SHA256:
    raise RuntimeError(
        f"Unexpected upstream DipDECK source: expected {EXPECTED_UPSTREAM_DIPDECK_SHA256}, "
        f"got {upstream_sha256}"
    )
for old, new in (
    ("self.labels.to(device)", "self.labels = self.labels.to(device)"),
    ("self.cluster_centers.to(device)", "self.cluster_centers = self.cluster_centers.to(device)"),
    ("self.dip_matrix.to(device)", "self.dip_matrix = self.dip_matrix.to(device)"),
):
    if source_text.count(old) != 1:
        raise RuntimeError(f"CUDA patch target is absent or ambiguous: {old}")
    source_text = source_text.replace(old, new)
dipdeck_source.write_text(source_text, encoding="utf-8")
patched_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
if patched_sha256 != EXPECTED_DIPDECK_SHA256:
    raise RuntimeError(
        f"Patched DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {patched_sha256}"
    )

repo_commit = "project-folder"

for relative, expected in DATA_SHA256.items():
    path = repo / relative
    if not path.is_file():
        raise FileNotFoundError(f"Project copy is missing {relative}")
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    if observed != expected:
        raise RuntimeError(f"{relative}: expected SHA256 {expected}, got {observed}")
if (repo / "data/phosphatase/phosphatase/data_standard.parquetbundle").is_file() is False:
    raise FileNotFoundError("Project copy is missing the phosphatase annotation bundle")

os.chdir(repo)
print(f"Repository: {repo}")
print(f"Project copied from: {PROJECT_DIR}")
print(f"Data checked: {list(DATA_SHA256)}")
print(f"ClustPy revision: {CLUSTPY_COMMIT}")

In [ ]:
COLAB_PINS = {
    "Cython": "3.3.0",
    "numpy": "2.1.3",
    "scipy": "1.18.1",
    "scikit-learn": "1.6.1",
    "pandas": "2.2.3",
    "h5py": "3.16.0",
    "pyarrow": "25.0.1",
    "tqdm": "4.70.0",
    "pytest": "9.1.1",
    "psutil": "7.2.2",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
}

import os
import subprocess
import sys

os.chdir("/content")
install_names = [
    "Cython", "numpy", "scipy", "scikit-learn", "pandas",
    "h5py", "pyarrow", "tqdm", "pytest", "psutil",
]
requirements = [f"{name}=={COLAB_PINS[name]}" for name in install_names]
install_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", *requirements],
    check=True, capture_output=True, text=True,
)
print(install_result.stdout)
if install_result.stderr:
    print(install_result.stderr, file=sys.stderr)

if "torch" in sys.modules or "torchvision" in sys.modules:
    raise RuntimeError("torch is already imported. Use Runtime > Restart session, then run all cells.")
torch_result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        f"torch=={COLAB_PINS['torch']}", f"torchvision=={COLAB_PINS['torchvision']}",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ],
    check=True, capture_output=True, text=True,
)
print(torch_result.stdout[-2000:])
if torch_result.stderr:
    print(torch_result.stderr[-2000:], file=sys.stderr)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "--force-reinstall", "--no-cache-dir", "--editable", str(clustpy_source),
] , check=True)

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True,
)
print(pip_check.stdout)
if pip_check.stderr:
    print(pip_check.stderr, file=sys.stderr)
if pip_check.returncode != 0:
    print("pip check reported the Colab base-environment warning above.", file=sys.stderr)

In [ ]:
import hashlib
import importlib
import importlib.metadata as metadata
import json
import os
import platform
import sys
from pathlib import Path

for module_name in (
    "Cython", "h5py", "pandas", "psutil", "pyarrow", "pytest", "scipy", "sklearn", "tqdm",
):
    importlib.import_module(module_name)

import numpy as np
import torch
import torchvision
from threadpoolctl import threadpool_info

if not (clustpy_source / "clustpy/__init__.py").is_file():
    raise FileNotFoundError(f"ClustPy package is missing from the verified checkout: {clustpy_source}")
sys.path.insert(0, str(clustpy_source))
import clustpy
from clustpy.deep import DipDECK
if Path(clustpy.__file__).resolve().parent != (clustpy_source / "clustpy").resolve():
    raise RuntimeError(f"Imported ClustPy is not the verified checkout: {clustpy.__file__}")

observed_versions = {name: metadata.version(name) for name in COLAB_PINS}
version_mismatches = {
    name: {"expected": expected, "observed": observed_versions[name]}
    for name, expected in COLAB_PINS.items()
    if observed_versions[name] != expected
}
if version_mismatches:
    raise RuntimeError(f"Colab dependency mismatch: {version_mismatches}")

dipdeck_source = clustpy_source / "clustpy/deep/dipdeck.py"
dipdeck_sha256 = hashlib.sha256(dipdeck_source.read_bytes()).hexdigest()
if dipdeck_sha256 != EXPECTED_DIPDECK_SHA256:
    raise RuntimeError(
        f"DipDECK source mismatch: expected {EXPECTED_DIPDECK_SHA256}, got {dipdeck_sha256}"
    )

environment = {
    "python": platform.python_version(),
    "project_commit": repo_commit,
    "clustpy_version": metadata.version("clustpy"),
    "clustpy_path": clustpy.__file__,
    "clustpy_commit": CLUSTPY_COMMIT,
    "dipdeck_sha256": dipdeck_sha256,
    "package_versions": observed_versions,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_num_threads": torch.get_num_threads(),
    "torch_num_interop_threads": torch.get_num_interop_threads(),
    "thread_environment": {
        name: os.environ.get(name)
        for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    },
    "threadpool_info": threadpool_info(),
}
print(json.dumps(environment, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU, then run all again.")

In [ ]:
import hashlib
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

standalone = repo / "src/algo/dipdeck/dipdeck_standalone.py"
if not standalone.is_file():
    raise FileNotFoundError(f"Standalone is missing: {standalone}")
output_root = (drive_root / "results/dipdeck_esm2_3b_seeds0to4").resolve()
output_root.mkdir(parents=True, exist_ok=True)
standalone_env = os.environ.copy()
standalone_env["DIPDECK_OUTPUT_ROOT"] = str(output_root)
completed_jobs = []

JOBS = {
    "raw-reference": (
        "scientific_reference_esm2_3b_raw2560", "--run-scientific-reference",
        "--validate-scientific-reference-output", None,
    ),
    "raw-grid": (
        "scientific_grid_esm2_3b_raw2560", "--run-scientific",
        "--validate-scientific-output", None,
    ),
    "pca-reference": (
        f"scientific_reference_esm2_3b_pca{PCA_DIM}", "--run-scientific-pca-reference",
        "--validate-scientific-pca-reference-output", PCA_DIM,
    ),
    "pca-grid": (
        f"scientific_grid_esm2_3b_pca{PCA_DIM}", "--run-scientific-pca",
        "--validate-scientific-pca-output", PCA_DIM,
    ),
}


def run_job(name: str) -> None:
    folder, run_flag, validate_flag, pca_dim = JOBS[name]
    owned_directory = output_root / folder
    shared = ["--embedding", EMBEDDING, "--seeds", SEEDS]
    if pca_dim is not None:
        shared += ["--pca-dim", str(pca_dim)]
    command = [
        sys.executable, str(standalone), run_flag, *shared,
        "--device", "cuda", "--clustpy-source-root", str(clustpy_source),
    ]
    validator = [
        sys.executable, str(standalone), validate_flag, *shared,
        "--output-root", str(output_root),
    ]
    archive_base = drive_root / f"dipdeck_{folder}_seeds0to4_outputs"
    archive_path = archive_base.with_suffix(".zip")
    already_valid = False
    if owned_directory.exists():
        already_valid = subprocess.run(validator, cwd=repo).returncode == 0
        if not already_valid:
            print(f"Replacing incomplete output: {owned_directory}")
    if already_valid:
        print(f"SKIP {folder}: output on Drive already validates.")
    else:
        subprocess.run(command, cwd=repo, env=standalone_env, check=True)
        subprocess.run(validator, cwd=repo, check=True)
    temporary_base = Path(f"{archive_base}_partial")
    temporary_zip = temporary_base.with_name(temporary_base.name + ".zip")
    temporary_zip.unlink(missing_ok=True)
    shutil.make_archive(str(temporary_base), "zip", root_dir=output_root, base_dir=folder)
    expected_files = sum(1 for path in owned_directory.rglob("*") if path.is_file())
    with zipfile.ZipFile(temporary_zip) as packed:
        if packed.testzip() is not None:
            raise RuntimeError(f"Corrupt member in {temporary_zip}")
        packed_files = sum(1 for name in packed.namelist() if not name.endswith("/"))
    if packed_files != expected_files:
        raise RuntimeError(f"{temporary_zip} holds {packed_files} files, folder has {expected_files}")
    os.replace(temporary_zip, archive_path)
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    completed_jobs.append((folder, archive_path.name, digest))
    print(f"Output archive: {archive_path}")
    print(f"Output SHA256: {digest}")


In [ ]:
run_job("raw-reference")


In [ ]:
run_job("raw-grid")


In [ ]:
run_job("pca-reference")


In [ ]:
run_job("pca-grid")


In [ ]:
print(f"{'job':<36} {'archive':<52} sha256")
for job, archive, digest in completed_jobs:
    print(f"{job:<36} {archive:<52} {digest}")
